In [0]:
employee=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/workspace/employee_project/employee_files/employees_data_quality_test.csv")

In [0]:
employee.show()

+-----------+----------+---------+----------------+-----------+----------+-------+------------+----------+---------------+--------------------+
|employee_id|first_name|last_name|      department|designation|manager_id| salary|joining_date|      city|employee_status|               email|
+-----------+----------+---------+----------------+-----------+----------+-------+------------+----------+---------------+--------------------+
|  EMP100001|   Devansh|  Thakkar|           Sales|    Manager|      NULL| 578082|  2025-11-26|   Chennai|         Active|devansh.thakkar@c...|
|  EMP100002|   Yashoda|    Batra|           Sales|   Engineer| EMP100001| 372132|  2024-01-29|   Chennai|       Inactive|yashoda.batra@com...|
|  EMP100003|     Meera| Banerjee|            NULL|   Engineer| EMP100001| 598830|  2025-08-25| Bengaluru|         Active|meera.banerjee@co...|
|  EMP100004|   Agastya|    Chana|           Sales|   Engineer| EMP100001|1174244|  2021-12-16| Bengaluru|         Active|agastya.chana@

In [0]:
missing_department=employee.filter(col("department").isNull())
missing_department.show()

+-----------+----------+---------+----------+-----------+----------+------+------------+---------+---------------+--------------------+
|employee_id|first_name|last_name|department|designation|manager_id|salary|joining_date|     city|employee_status|               email|
+-----------+----------+---------+----------+-----------+----------+------+------------+---------+---------------+--------------------+
|  EMP100003|     Meera| Banerjee|      NULL|   Engineer| EMP100001|598830|  2025-08-25|Bengaluru|         Active|meera.banerjee@co...|
+-----------+----------+---------+----------+-----------+----------+------+------------+---------+---------------+--------------------+



In [0]:
missing_email=employee.filter(col("email").isNull())
missing_email.show()

+-----------+----------+---------+----------+-----------+----------+------+------------+-----+---------------+-----+
|employee_id|first_name|last_name|department|designation|manager_id|salary|joining_date| city|employee_status|email|
+-----------+----------+---------+----------+-----------+----------+------+------------+-----+---------------+-----+
|  EMP100013|     Inaya|     Dutt|        HR|    Manager| EMP100001|812262|  2025-12-28|Kochi|         Active| NULL|
+-----------+----------+---------+----------+-----------+----------+------+------------+-----+---------------+-----+



### understand what happens when we convert string to int below

In [0]:
employee.select(
    *[
        col(c).isNull().cast("int")
        for c in employee.columns
    ]
).show()

+----------------------------------+---------------------------------+--------------------------------+---------------------------------+----------------------------------+---------------------------------+-----------------------------+-----------------------------------+---------------------------+--------------------------------------+----------------------------+
|CAST((employee_id IS NULL) AS INT)|CAST((first_name IS NULL) AS INT)|CAST((last_name IS NULL) AS INT)|CAST((department IS NULL) AS INT)|CAST((designation IS NULL) AS INT)|CAST((manager_id IS NULL) AS INT)|CAST((salary IS NULL) AS INT)|CAST((joining_date IS NULL) AS INT)|CAST((city IS NULL) AS INT)|CAST((employee_status IS NULL) AS INT)|CAST((email IS NULL) AS INT)|
+----------------------------------+---------------------------------+--------------------------------+---------------------------------+----------------------------------+---------------------------------+-----------------------------+--------------------------

### Reason using sum of count because it is converted into 0 and 1 
- 0 not null
- 1 null 
sum up get to understand total number of null

## Total Number of Null values in each column

In [0]:
employee.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in employee.columns
    ]
).show()

+-----------+----------+---------+----------+-----------+----------+------+------------+----+---------------+-----+
|employee_id|first_name|last_name|department|designation|manager_id|salary|joining_date|city|employee_status|email|
+-----------+----------+---------+----------+-----------+----------+------+------------+----+---------------+-----+
|          0|         0|        0|         1|          0|         1|     0|           0|   1|              0|    1|
+-----------+----------+---------+----------+-----------+----------+------+------------+----+---------------+-----+



In [0]:
# Option 1 Fill the Missing Value
updated_null_column=employee.fillna({
    "department":"Unknown",
    "city":"Unknown"
})

In [0]:
updated_null_column.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in updated_null_column.columns
    ]
).show()

+-----------+----------+---------+----------+-----------+----------+------+------------+----+---------------+-----+
|employee_id|first_name|last_name|department|designation|manager_id|salary|joining_date|city|employee_status|email|
+-----------+----------+---------+----------+-----------+----------+------+------------+----+---------------+-----+
|          0|         0|        0|         0|          0|         1|     0|           0|   0|              0|    1|
+-----------+----------+---------+----------+-----------+----------+------+------------+----+---------------+-----+



In [0]:
# Option 2 drop record whose city is null
drop_records=employee.dropna(
    subset=["city"]
)
drop_records.show()

+-----------+----------+---------+----------------+-----------+----------+-------+------------+----------+---------------+--------------------+
|employee_id|first_name|last_name|      department|designation|manager_id| salary|joining_date|      city|employee_status|               email|
+-----------+----------+---------+----------------+-----------+----------+-------+------------+----------+---------------+--------------------+
|  EMP100001|   Devansh|  Thakkar|           Sales|    Manager|      NULL| 578082|  2025-11-26|   Chennai|         Active|devansh.thakkar@c...|
|  EMP100002|   Yashoda|    Batra|           Sales|   Engineer| EMP100001| 372132|  2024-01-29|   Chennai|       Inactive|yashoda.batra@com...|
|  EMP100003|     Meera| Banerjee|            NULL|   Engineer| EMP100001| 598830|  2025-08-25| Bengaluru|         Active|meera.banerjee@co...|
|  EMP100004|   Agastya|    Chana|           Sales|   Engineer| EMP100001|1174244|  2021-12-16| Bengaluru|         Active|agastya.chana@

In [0]:
print(f"Before dropping a missing city:{employee.select("department").count()}")
print(f"After dropping a missing city:{drop_records.select("department").count()}")

Before dropping a missing city:26
After dropping a missing city:25
